# BIOT 6900 · Module 2 · Assignment 2 — Pancreatic Cancer (CPTAC-PDAC)

Independent multi-omics target discovery. Built on the gene-level schema from Part 3
(`gene, log2fc, pval` per layer -> harmonize -> concordance -> multi-evidence score -> rank -> export),
applied to real CPTAC-PDAC data instead of synthetic/Alzheimer's data.

**Data.** CPTAC-PDAC (LinkedOmics), open access, no data-use agreement:
https://www.linkedomics.org/data_download/CPTAC-PDAC/
- RNA: `mRNA_RSEM_UQ_log2_Tumor.cct` / `mRNA_RSEM_UQ_log2_Normal.cct` (140 tumor / 21 normal)
- Protein: `proteomics_gene_level_MD_abundance_tumor.cct` / `..._normal.cct` (140 tumor / 75 normal)
- Mutation: `Mutation_gene_level.cgt` (140 tumor samples, binary mutation status per gene)

**Note on "unmatched" framing.** Unlike the Alzheimer's assignment, these files ARE sample-matched
(same 140 tumors across RNA + protein + mutation). We still reduce to one row per gene -- computing a
tumor-vs-normal differential (log2fc + p-value) per gene -- so the downstream schema matches the
taught pipeline exactly. This is worth stating explicitly in your report's "matched vs. gene-level"
discussion.

In [1]:
import os
import numpy as np
import pandas as pd
from scipy import stats

EQUAL_WEIGHTS = {"transcriptomic": 1/3, "proteomic": 1/3, "genomic": 1/3}
DATA_DIR = "data_pdac" 

## Helper functions (identical to Part 1/3 -- do not need to edit)

In [2]:
mut_raw = pd.read_csv(f"{DATA_DIR}/Mutation_gene_level.cgt", sep="\t", index_col=0)
print(mut_raw.shape)
print(mut_raw.iloc[:5, :5])
print("Sample of unique values in the table:")
print(pd.unique(mut_raw.values.ravel())[:20])

(4424, 140)
      C3L-03394 C3N-03428 C3L-02112 C3N-01719 C3N-03670
A1BG         WT        WT        WT        WT        WT
A2ML1        WT        WT        WT        WT        WT
A4GNT        WT        WT        WT        WT        WT
AAGAB        WT        WT        WT        WT        WT
AARS         WT        WT        WT        WT        WT
Sample of unique values in the table:
['WT' 'missense_variant' 'missense_variant,splice_region_variant'
 'synonymous_variant' 'frameshift_variant'
 'missense_variant;missense_variant'
 'frameshift_variant,splice_region_variant' 'splice_acceptor_variant'
 'splice_donor_variant,non_coding_transcript_variant' 'stop_gained'
 'inframe_deletion' 'synonymous_variant;missense_variant'
 'stop_gained,splice_region_variant' 'splice_donor_variant'
 'missense_variant;synonymous_variant;missense_variant'
 'stop_gained,frameshift_variant'
 'missense_variant;synonymous_variant;synonymous_variant'
 'inframe_insertion' 'missense_variant;synonymous_variant'
 'syn

In [3]:
def rank_percentile(series: pd.Series) -> pd.Series:
    """Normalize any score to [0, 1] by rank. Robust to outliers and scale differences."""
    return series.rank(method="average", pct=True)


def multi_evidence_score(df: pd.DataFrame, cols, weights) -> pd.Series:
    """Weighted sum of rank-percentile-normalized layer scores."""
    normed = pd.DataFrame({c: rank_percentile(df[c]) for c in cols})
    return sum(weights[c] * normed[c] for c in cols)

## 3.1 -- Load and reduce each layer to one row per gene

Each raw file is a gene x sample matrix. We reduce each to `gene, log2fc, pval` (RNA, protein) or
`gene, neglog10p` (genomics) -- exactly the schema the rubric grades against.

**Check the printed shapes.** If a matrix looks transposed (~140 rows instead of ~20,000+), add
`.T` after that `pd.read_csv(...)` call.

In [4]:
def load_layer(tumor_path, normal_path):
    t = pd.read_csv(tumor_path, sep="\t", index_col=0)
    n = pd.read_csv(normal_path, sep="\t", index_col=0)
    return t, n

rna_t, rna_n = load_layer(f"{DATA_DIR}/mRNA_RSEM_UQ_log2_Tumor.cct",
                           f"{DATA_DIR}/mRNA_RSEM_UQ_log2_Normal.cct")
prot_t, prot_n = load_layer(f"{DATA_DIR}/proteomics_gene_level_MD_abundance_tumor.cct",
                             f"{DATA_DIR}/proteomics_gene_level_MD_abundance_normal.cct")
mut_t = pd.read_csv(f"{DATA_DIR}/Mutation_gene_level.cgt", sep="\t", index_col=0)
mut_t = (mut_t.notna() & (mut_t != "WT")).astype(int)

print("RNA tumor:", rna_t.shape, " RNA normal:", rna_n.shape)
print("Protein tumor:", prot_t.shape, " Protein normal:", prot_n.shape)
print("Mutation:", mut_t.shape, "(gene x sample, binary)")

RNA tumor: (28057, 140)  RNA normal: (28057, 21)
Protein tumor: (11662, 140)  Protein normal: (11662, 75)
Mutation: (4424, 140) (gene x sample, binary)


In [5]:
def differential_table(tumor_df, normal_df, lfc_col, p_col):
    """Per-gene tumor-vs-normal log2FC + t-test p-value -> tidy gene table."""
    common_genes = tumor_df.index.intersection(normal_df.index)
    t = tumor_df.loc[common_genes]
    n = normal_df.loc[common_genes]
    lfc = t.mean(axis=1) - n.mean(axis=1)
    pval = pd.Series(
        {g: stats.ttest_ind(t.loc[g], n.loc[g], equal_var=False, nan_policy="omit").pvalue
         for g in common_genes},
        name=p_col)
    out = pd.DataFrame({"gene": common_genes, lfc_col: lfc.values, p_col: pval.values})
    return out.dropna()

tx = differential_table(rna_t, rna_n, "log2fc", "pval")
pr = differential_table(prot_t, prot_n, "log2fc", "pval")
print(f"tx (transcriptomics): {tx.shape}")
print(f"pr (proteomics):      {pr.shape}")
tx.head()

C:\Users\USER\AppData\Local\Temp\ipykernel_21032\2907630620.py:8: SmallSampleWarning: After omitting NaNs, one or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  {g: stats.ttest_ind(t.loc[g], n.loc[g], equal_var=False, nan_policy="omit").pvalue


tx (transcriptomics): (26005, 3)
pr (proteomics):      (11630, 3)


,gene,log2fc,pval
0,A1BG,-0.479548,1.250690e-02
1,A1BG-AS1,-0.178861,4.499880e-01
2,A1CF,-1.463168,2.552527e-08
3,A2M,-0.624133,5.580447e-07
4,A2M-AS1,-0.480572,8.412509e-03


In [6]:
# Genomics Layer: mutation frequency -> enrichment p-value per gene (binomial test vs.
# genome-wide background mutation rate), then -log10(p) as association strength.
mut_freq = mut_t.mean(axis=1, skipna=True).dropna()
n_samples = mut_t.shape[1]
background_rate = mut_freq.mean()

def mut_pval(freq, n=n_samples, bg=background_rate):
    k = round(freq * n)
    return stats.binomtest(k, n, bg, alternative="greater").pvalue

gw = pd.DataFrame({
    "gene": mut_freq.index,
    "mut_freq": mut_freq.values,
    "neglog10p": [-np.log10(max(mut_pval(f), 1e-300)) for f in mut_freq.values],
})
print(f"gw (genomics): {gw.shape}")
gw.sort_values("neglog10p", ascending=False).head()

gw (genomics): (4424, 3)


,gene,mut_freq,neglog10p
1967,KRAS,0.964286,260.757865
3933,TP53,0.750000,176.571264
617,CDKN2A,0.207143,28.398510
3552,SMAD4,0.178571,22.895919
4035,TTN,0.114286,11.806851


## 3.2 -- Harmonize on gene symbol -> one joined table

In [7]:
tx2 = tx.rename(columns={"log2fc": "rna_lfc", "pval": "rna_p"})
pr2 = pr.rename(columns={"log2fc": "prot_lfc", "pval": "prot_p"})

df = tx2.merge(pr2, on="gene", how="inner").merge(gw, on="gene", how="inner")
print(f"Genes surviving the 3-way join: {len(df)} "
      f"(RNA {len(tx2)}, protein {len(pr2)}, genomic {len(gw)})")
df.head()

Genes surviving the 3-way join: 2532 (RNA 26005, protein 11630, genomic 4424)


,gene,rna_lfc,rna_p,prot_lfc,prot_p,mut_freq,neglog10p
0,A1BG,-0.479548,1.250690e-02,0.303070,2.094465e-04,0.007143,0.119799
1,A2ML1,3.030196,2.441368e-12,0.884157,2.971118e-08,0.007143,0.119799
2,A4GNT,-0.688471,6.228432e-02,-0.137812,4.689057e-01,0.007143,0.119799
3,AAGAB,0.321240,4.493614e-07,0.179743,6.447061e-04,0.007143,0.119799
4,AARS,-0.127723,1.951716e-01,-0.613124,1.002847e-13,0.007143,0.119799


## 3.3 -- Sign-agreement concordance
Does each gene move the same direction at RNA and protein (tumor vs. normal)?

In [8]:
df["concordant"] = np.sign(df["rna_lfc"]) == np.sign(df["prot_lfc"])
print(f"Sign-concordant genes: {df['concordant'].sum()} / {len(df)}")

Sign-concordant genes: 1650 / 2532


## 3.4 -- Multi-evidence score

In [9]:
df["transcriptomic"] = df["rna_lfc"].abs()
df["proteomic"] = df["prot_lfc"].abs()
df["genomic"] = df["neglog10p"]
df["score"] = multi_evidence_score(df, ["transcriptomic", "proteomic", "genomic"], EQUAL_WEIGHTS)
print("scored.")

scored.


## 3.5 -- Rank, inspect, export

Known PDAC drivers to check against: **KRAS, TP53, SMAD4, CDKN2A** (the classic four), plus
CA19-9-related genes (e.g. MUC1) and any others worth a second look.

In [10]:
ranked = df.sort_values("score", ascending=False)
top = ranked.head(15)
ranked.to_csv("targets_pdac.csv", index=False)
print(top[["gene", "rna_lfc", "prot_lfc", "neglog10p", "concordant", "score"]]
      .round(3).to_string(index=False))

   gene  rna_lfc  prot_lfc  neglog10p  concordant  score
COL11A1    2.787     1.448      1.263        True  0.984
  MUC5B    2.071     1.224      2.499        True  0.976
   TNS4    4.351     1.234      0.771        True  0.975
  ERBB4   -2.565     1.329      0.771       False  0.973
  PLIN4   -3.068    -1.197      0.771        True  0.973
  MUC16    3.162     0.922      6.636        True  0.971
    FN1    1.826     1.263      1.263        True  0.971
  POSTN    1.743     1.353      1.263        True  0.971
   NOS1   -1.916    -1.252      0.771        True  0.965
 MUC5AC    3.751     0.815      3.220        True  0.962
    CEL   -6.093    -2.712      0.383        True  0.954
  BRSK2   -2.976    -1.807      0.383        True  0.948
  MAT1A   -2.832    -1.966      0.383        True  0.948
  KRT6A    5.212     1.465      0.383        True  0.948
  FRAS1   -1.241    -1.155      0.771        True  0.945


## 3.6 -- Interpretation (write this yourself -- goes in your 3-4 page report)

1. **Disease & data choice.** Why PDAC? CPTAC-PDAC via LinkedOmics is open access, no data-use
   agreement. Note this data is sample-matched (140 tumors profiled at all three layers) even
   though we reduced it to gene-level differentials to match the pipeline schema -- state that
   explicitly, since it affects what you can claim (see point 4).
I chose pancreatic ductal adenocarcinoma (PDAC) because it is one of the most aggressive and lethal solid tumors, making the identification of potential therapeutic targets especially important. I used the CPTAC-PDAC dataset through LinkedOmics because it is openly available and does not require a data-use agreement.

The original dataset is sample-matched, with 140 tumor samples profiled using RNA-seq, proteomics, and somatic mutation data. There were also matched normal samples for the RNA and protein datasets. However, for this analysis, I reduced each data type to gene-level tumor-versus-normal summary statistics, such as log2 fold-change and p-values for RNA and protein, and mutation frequency and enrichment p-values for the genomic data. I made this choice to keep the three data types in a consistent format for the pipeline we were using. Therefore, although the original data are patient-matched, my analysis does not directly use the patient-level pairing.
2. **Weighting.** Equal weights were used here. Argue for keeping them, or justify a change
   (e.g., should mutation enrichment be weighted more heavily than expression change, since
   mutations are more likely to be causal rather than downstream?).
I kept equal weights, with transcriptomic, proteomic, and genomic evidence each contributing one-third to the final score.

There is a reasonable argument for giving more weight to the genomic layer because somatic mutations can be closer to the underlying cause of cancer-related changes. A mutation can lead to changes in RNA expression and protein levels, whereas an expression change could be a downstream effect of the tumor or its environment.

However, mutation frequency alone does not tell us whether a mutation has a strong functional effect. A frequently mutated gene may not show a corresponding change at the RNA or protein level, while a gene with a large expression or protein change could be biologically important even if it is not frequently mutated.

For that reason, I kept the weights equal. Without a specific biological or statistical justification for prioritizing one layer, equal weighting seemed like the least arbitrary approach and allowed each type of evidence to contribute to the final score.
3. **Top targets.** Which known PDAC genes did you recover (KRAS, TP53, SMAD4, CDKN2A, MUC1 ...)?
   Any non-obvious hit worth a second look -- check it against Open Targets or recent literature.
In the mutation-frequency analysis, I recovered the major known PDAC driver genes in the expected order: KRAS (96.4% mutation frequency), TP53 (75.0%), CDKN2A (20.7%), and SMAD4 (17.9%). This was useful because it showed that the genomic layer was identifying biologically established PDAC genes.

Interestingly, these genes were not all among the highest-ranked genes in the combined multi-omics score. The top combined hits included COL11A1, MUC5B, TNS4, ERBB4, PLIN4, MUC16, FN1, POSTN, NOS1, MUC5AC, CEL, BRSK2, MAT1A, KRT6A, and FRAS1.

One group that stood out was the mucin-related genes, including MUC16, MUC5B, and MUC5AC. COL11A1, FN1, and POSTN also stood out because they are associated with the extracellular matrix and stromal environment. These results are interesting in the context of PDAC because the tumor has a prominent stromal component.

A particularly interesting non-obvious hit was ERBB4 because it appeared among the top combined hits while also showing disagreement between the RNA and protein measurements. This makes it worth investigating further rather than assuming that a high combined score automatically means it is a strong therapeutic target. A literature or Open Targets check would be useful to determine how much independent evidence exists for ERBB4 in PDAC.
4. **Discordant gene.** Pick a `concordant == False` gene in your top hits (strong RNA, flat/
   opposite protein). What biology could explain RNA and protein disagreeing (post-transcriptional
   regulation, protein turnover, degradation)?
ERBB4 was one of the top genes where the RNA and protein measurements were discordant. The RNA showed a strong change, while the protein did not change in the same direction.

One possible explanation is post-transcriptional regulation. Changes in RNA abundance do not always lead to proportional changes in protein production. Regulatory mechanisms such as microRNAs can affect translation and reduce the amount of protein produced from a transcript.

Another explanation could be protein turnover. Even if the amount of ERBB4 RNA increases, the protein could be degraded more quickly through cellular protein-degradation pathways. Differences in the half-lives of RNA and protein could also contribute to the mismatch.

Therefore, the disagreement between RNA and protein does not necessarily indicate a data problem. It could reflect biological regulation between transcription and the final protein level. This is also why looking at both transcriptomic and proteomic data can provide information that would be missed by using RNA alone
5. **Limitation.** Because your underlying data was sample-matched but you integrated at the
   gene level (tumor-vs-normal summary statistics, not per-patient pairing), what can you claim
   about population-level trends -- and what can't you claim about any individual patient or about
   causality?
The main limitation is that, even though the original CPTAC-PDAC dataset contains sample-matched measurements, my analysis used gene-level summary statistics rather than the paired measurements from each individual patient.

Because of this, I can make conclusions about population-level or cohort-level trends. For example, I can say that MUC16 expression is elevated on average in PDAC tumor samples compared with normal samples in this cohort.

However, I cannot use this analysis to predict what the MUC16 expression level would be for a specific individual patient. Individual patients may have substantial biological differences that are hidden when the data are summarized at the cohort level.

I also cannot make causal claims from these results. A gene being frequently mutated or having a large RNA or protein change does not prove that it is causing tumor development or progression. The results show associations within this dataset and can be used to identify candidates for further investigation, but additional experimental or patient-level analysis would be needed to establish causality.

Overall, the biggest takeaway is that the original dataset provides rich patient-matched multi-omics information, but reducing it to gene-level summary statistics makes the analysis simpler and more consistent with the pipeline while also limiting the types of conclusions that can be drawn.
## Submit
1. `Kernel -> Restart & Run All`
2. Commit this notebook, `targets_pdac.csv`, and a README (name, disease, each dataset + source
   link + access type) to your `biot6900` repo.
3. Push, confirm files on GitHub, post the repo link on Canvas.3